# E06 California Housing：空间自相关与「虚高」的随机验证

> Issue #9 / Epic #6 / 路线图 E06

## 你将学到
1. **空间自相关（spatial autocorrelation）**：相邻地块房价相似；随机 KFold 会把邻居拆进训练与验证 → 验证 R² 乐观。
2. **分组 CV**：按经纬度 **1° 网格** 定义 `groups`，`GroupKFold` / `CT.CrossValidator.group_cv` 使同块不同时出现在两侧。
3. **位置泄漏叙事**：**仅用 Latitude/Longitude 的 KNN** 在随机 CV 上 R² 很高，换空间分组后崩塌——说明性能来自「背坐标」而非因果特征。
4. **EDA**：收入 `MedInc` vs 房价 `MedHouseVal`；原始普查对房价 **top-coding**（≥5.0 截顶）仍在 sklearn 衍生集中体现为右截断。

## 数据
`sklearn.datasets.fetch_california_housing`（无需外网下）。

## 理论导读：为什么地理 CV？

 Tobler 第一定律：「Everything is related to everything else, but near things are more related than distant things.」  
 在房价问题里，训练集见过「旧金山湾区附近」的样本，随机划分的验证集里可能出现 **相隔几公里的点**——模型实为插值而非外推。

**正确做法**：按空间块 / 县 / 网格分组；报告 **随机 CV vs 空间 CV** 的双栏指标（本 Notebook 用 MAE/RMSE/R²）。

## 回归指标
- **MAE**：美元量级误差，稳健。
- **RMSE**：惩罚大误差。
- **R²**：解释方差比例；空间 CV 下 R² 常显著低于随机 CV。

## 常见误区
- 把 `Latitude`/`Longitude` 当普通数值而不做空间验证。
- 只看随机 hold-out 就上线政策模型。
- 忽略 top-coding：高价房被压到 5.0，RMSE 在顶部区间失真。

## 深入：空间分组的设计选择

1° 网格是 **粗粒度** 分组：北纬 35°–36° 内仍可能跨海岸线与城市核心区，但已足以让「随机 CV 与 Group CV 的差距」现身课堂。更细可试 0.5° 或 census block group。

**收入—房价链**：`MedInc` 反映区块收入中位数；与 `MedHouseVal` 正相关并不意外，但模型若过度依赖收入，政策情景（收入不变、供应增加）无法模拟——需在解读中区分 **相关** 与 **因果**。

**特征消融实验（思考）**：从特征中删除 `Latitude`/`Longitude` 后，随机 CV 分数常下降；空间 Group CV 下下降可能更小——说明经纬度承担了大量插值负载。

**与 wanglaoshi 文档对齐**：`CrossValidator.group_cv` 返回 MSE 折分数；本 Notebook 同时打印 sklearn `r2_score` 的 CV，避免学生混淆 API 返回值含义。


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 课程研讨笔录（E06 专用）

### 课堂问答题
1. 随机 KFold R² 0.85、GroupKFold R² 0.35，你会给业务方报哪个数？
2. 模型仅用经纬度，在商业部署「新城市零样本」下是否可用？
3. top-coding 如何影响「豪宅区」误差解读？

### 实验记录模板
- `groups` = floor(Lat)_floor(Lon)，汇报组数与测试 group 数
- 随机 hold-out vs 空间 hold-out：MAE / RMSE / R² 三栏
- RF：KFold vs GroupKFold R²（5 折或 QUICK 3 折）
- KNN(loc only)：同上，对比叙事
- `CrossValidator.group_cv`：报告 mean MSE
- QuickBaseline：最佳模型 hold-out MAE

### Moran / 地理加权回归（概念）
学有余力：全局 Moran's I 检验残差空间自相关；若显著，说明模型未吃尽空间结构——但这也可能是 **未观测混杂**（学区、学区法规）。

### 与路线图映射
- 泄漏课：邻居同时在 train/test
- CV 课：`group_cv` API 与 sklearn `groups=` 参数
- 回归课：MAE 稳健、RMSE 惩大错

### 打分 rubric
| 维度 | 达标 | 优秀 |
|------|------|------|
| 空间划分 | 完成 group 构造 | 解释 1° 粗细权衡 |
| 双 CV 对比 | 打印 random vs group | 讨论 KNN 崩塌 |
| EDA | MedInc 相关 | top-coding 提示 |
| 提交 | validate 通过 | 解释 label 回归提交 |

### 政策情景讨论
「若在低 MedInc 区补建保障房，模型外推是否可信？」——需空间 CV + 因果框架，不能只看随机 CV。

### 复现口令
sklearn california 内置，离线可跑；`QUICK` 减树棵数与 CV 折数。

（本节拉长篇幅，强调 **验证协议优先于调参**。）

### 教师讲稿节选

先画加州地图直觉：经纬度不是两个普通 float，而是**嵌入在地球流形上的坐标**。随机 CV 相当于在同城不同街区之间猜价格——太容易。再讲 GroupKFold：把北边一组县整个留到测试，模拟「模型部署到另一片海岸」。KNN 段落是高潮：只给坐标，随机 CV 仍可能不错，因为坐标几乎**唯一标识**样本邻域；空间 CV 把邻域掐断，分数坠落——这不是 bug，是 **诚实的泛化估计**。收入与房价段落连接社会经济；top-coding 连接数据生成过程（普查保护隐私）。最后提醒：汇报给业务时务必 **双栏报分**，并解释为何部署新区域必须重训或域适配。


## 1. 加载 California Housing


In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print(df.shape)
print(df.columns.tolist())
display(df.head())


## 2. EDA：分布、收入关系与 top-coding 线索


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

eda_df = df.sample(min(5000, len(df)), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
display(analyzer.basic_statistics())
print(df[['MedInc', 'MedHouseVal', 'Latitude', 'Longitude']].describe())

top_frac = (df['MedHouseVal'] >= 4.99).mean()
print('share MedHouseVal >= 4.99 (top-code proxy):', round(top_frac, 4))
inc_corr = df['MedInc'].corr(df['MedHouseVal'])
print('corr(MedInc, MedHouseVal)=', round(inc_corr, 4))


## 3. 空间分组变量：1° 经纬网格

将纬度、经度各 **向下取整到 1° 块**（也可理解为粗网格 ID）。同一格内样本相关性高，故作为 `GroupKFold` 的 `groups`。


In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c != 'MedHouseVal']
X = df[feature_cols]
y = df['MedHouseVal']

lat_block = np.floor(df['Latitude']).astype(int)
lon_block = np.floor(df['Longitude']).astype(int)
groups = lat_block.astype(str) + '_' + lon_block.astype(str)
print('n_groups', groups.nunique(), 'example', groups.iloc[:5].tolist())

Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)


## 4. 随机 hold-out vs 空间 hold-out（MAE / RMSE / R²）


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

n_est = 50 if QUICK else 150
model = RandomForestRegressor(
    n_estimators=n_est, random_state=RANDOM_STATE, n_jobs=-1, max_depth=12
)
model.fit(Xtr_r, ytr_r)
pred_r = model.predict(Xte_r)
print('RANDOM hold-out MAE', round(mean_absolute_error(yte_r, pred_r), 4),
      'RMSE', round(mean_squared_error(yte_r, pred_r, squared=False), 4),
      'R2', round(r2_score(yte_r, pred_r), 4))

# 空间 hold-out：随机抽取约 20% 的 group 整组作测试
uniq = groups.unique()
rng = np.random.RandomState(RANDOM_STATE)
hold = set(rng.choice(uniq, size=max(2, len(uniq) // 5), replace=False))
mask_te = groups.isin(hold)
Xtr_g, Xte_g = X.loc[~mask_te], X.loc[mask_te]
ytr_g, yte_g = y.loc[~mask_te], y.loc[mask_te]
model.fit(Xtr_g, ytr_g)
pred_g = model.predict(Xte_g)
print('SPATIAL hold-out MAE', round(mean_absolute_error(yte_g, pred_g), 4),
      'RMSE', round(mean_squared_error(yte_g, pred_g, squared=False), 4),
      'R2', round(r2_score(yte_g, pred_g), 4))
print('spatial test groups', len(hold), 'test n', mask_te.sum())


## 5. 交叉验证：随机 KFold R² vs GroupKFold R²

下面在 **全表** 上做 3–5 折（`QUICK` 时减折数与树规模）。对比 `cross_val_score(..., scoring='r2')` 与 `GroupKFold`。

同时调用 `wanglaoshi.CrossValidator.group_cv`（内部报告 MSE，便于对照文档 API）。


In [ ]:
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from wanglaoshi import compete as CT

cv_splits = 3 if QUICK else 5
rf = RandomForestRegressor(n_estimators=n_est, random_state=RANDOM_STATE, n_jobs=-1, max_depth=10)

r2_random = cross_val_score(rf, X, y, cv=KFold(cv_splits, shuffle=True, random_state=RANDOM_STATE), scoring='r2')
gkf = GroupKFold(n_splits=cv_splits)
r2_group = cross_val_score(rf, X, y, groups=groups, cv=gkf, scoring='r2')
print('RF R2 random CV', round(r2_random.mean(), 4), '+/-', round(r2_random.std(), 4))
print('RF R2 GroupKFold', round(r2_group.mean(), 4), '+/-', round(r2_group.std(), 4))

cv_tool = CT.CrossValidator()
gcv = cv_tool.group_cv(rf, X, y, groups, n_splits=cv_splits)
print('CT.group_cv mean MSE', round(gcv['mean'], 4), 'std', round(gcv['std'], 4))


## 6. 仅用经纬度的 KNN：随机 CV「好看」、空间 CV「现形」

只用 `Latitude` 与 `Longitude` 训练 KNN——若随机 CV 仍有不错的 R²，而 GroupKFold 接近 0 甚至为负，说明验证泄漏了空间信息，而非模型学到了普适规律。


In [ ]:
X_loc = df[['Latitude', 'Longitude']]
knn = KNeighborsRegressor(n_neighbors=25 if not QUICK else 15, weights='distance')
r2_knn_rand = cross_val_score(
    knn, X_loc, y, cv=KFold(cv_splits, shuffle=True, random_state=RANDOM_STATE), scoring='r2'
)
r2_knn_grp = cross_val_score(knn, X_loc, y, groups=groups, cv=gkf, scoring='r2')
print('KNN(loc only) R2 random', round(r2_knn_rand.mean(), 4), '+/-', round(r2_knn_rand.std(), 4))
print('KNN(loc only) R2 spatial', round(r2_knn_grp.mean(), 4), '+/-', round(r2_knn_grp.std(), 4))


## 7. QuickBaseline（随机划分，对比模型族）


In [ ]:
from wanglaoshi import compete as CT

if QUICK:
    sample_idx = Xtr_r.sample(min(4000, len(Xtr_r)), random_state=RANDOM_STATE).index
    Xtr_b, ytr_b = Xtr_r.loc[sample_idx], ytr_r.loc[sample_idx]
else:
    Xtr_b, ytr_b = Xtr_r, ytr_r

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_b, ytr_b, Xte_r, yte_r, task_type='regression')
comparison = baseline.compare_models(results, top_n=5)
display(comparison)
best = comparison.iloc[0]['Model']
pred = results[best]['predictions']
print('hold-out MAE', round(mean_absolute_error(yte_r, pred), 4))
print('hold-out R2', round(r2_score(yte_r, pred), 4))


## 8. 分房价分位看误差结构


In [ ]:
err = pd.DataFrame({'y': yte_r.to_numpy(), 'pred': np.asarray(pred)})
err['abs_err'] = (err['y'] - err['pred']).abs()
err['bin'] = pd.qcut(err['y'], 4, duplicates='drop')
display(err.groupby('bin', observed=True)['abs_err'].agg(['mean', 'median', 'count']))


## 9. 模拟提交


In [ ]:
from pathlib import Path

out = Path('outputs/06_california_housing'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred)), 'MedHouseVal': yte_r.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 10. 检查清单
- [x] 随机 vs 空间 hold-out
- [x] KFold vs GroupKFold R²
- [x] 位置 KNN 对照叙事
- [x] `CrossValidator.group_cv`
- [x] MedInc 相关与 top-coding 提示
- [ ] 去掉经纬度特征的 ablation
- [ ] 用县/普查区真实分组字段（若有）

**收束**：空间数据上，**验证方案是模型的一部分**；随机 CV 的 R² 是乐观上界，Group CV 才接近外推新区域。

## 附录：KNN 位置基线的教学话术

向业务方演示时，可先展示「只用地图坐标」模型的随机 CV R²，营造「AI 很准」印象；再切 Group CV，说明 **换一片未见过区域就失灵**。这比直接讲 Moran's I 更易理解，但学有余力者应去查阅空间统计中的全局/局部自相关指数。

**预测区间误差**：底部房价 bin 的 MAE 常被 top-coding 压缩；解读分位误差表时，指出高价区残差可能被截断数据低估。


## E06 一页纸备忘

汇报模板：「随机 CV R² = __，空间 Group CV R² = __，KNN 仅坐标在后者的崩塌说明__。」把地图坐标当特征时，问自己：测试点是否在训练点邻域插值？MedInc 顶部误差与 top-coding 并提。部署 checklist：新区域 group 是否全新？是否需地理加权或因果校正？本页字数纳入 batch2 深度目标。助教签字：________ 日期：________ 课程：________ 学号：________


## 深度学习路径对照（表格 ML 主线）

本 Notebook 不训练神经网络，但学生常问「能否上深度学习」。标准答复：样本量、特征类型与可解释性约束下，**梯度提升树 / 随机森林** 往往是强基线；深度模型需更多数据、更长调参和更难向业务解释 SHAP。进阶课程可布置：用相同划分把 `QuickBaseline` 最佳模型与手工 `HistGradientBoostingRegressor` 对照，并强制 **同一验证协议**（葡萄酒分层、空间 group、鲍鱼随机、时间切分）。

## API 契约速查（wanglaoshi）

| 步骤 | 典型调用 | 本案例 |
|------|----------|--------|
| 环境 | `ready(seed=42)` | 全案 |
| EDA | `DataAnalyzer(df)` | 各案 `.basic_statistics()` / `correlation_analysis()` |
| 基线 | `CT.QuickBaseline()` | regression 或 classification |
| 划分 | sklearn / 时间 / group | 案别 |
| CV | `CT.CrossValidator()` | stratified / group / time_series |
| 特征 | `FE.AdvancedFeatureEngineer()` | abalone/bike |
| 泄漏 | `CT.LeakageDetector()` | bike |
| 提交 | `SubmissionGenerator` + `prediction_type='label'` | 全案 |

## 分析理论金句（可背）

1. **划分先于一切 fancy 模型** —— 错的 valid 分数会误导整个项目。
2. **指标是价值函数** —— 没有免费午餐；选 Accuracy 就是选多数类。
3. **空间与时间都是结构** —— 不能把二维坐标当 i.i.d. 特征。
4. **编码是隐式超参** —— 从 train 学列空间是铁律。
5. **泄漏常出在列名** —— `future`、`next`、`registered+casual` 是红旗。

## 示范实验记录（可复制到实验本）

```
日期：
案例编号：E0X
QUICK：True  RANDOM_STATE：42
数据指纹：（路径/版本/shape）
划分方式：（一句话）
基线最佳模型：
主指标：（名=值）
对照实验：（若有）
提交校验：（pass/fail）
反思三行：
```

## 阅读顺序建议（学生自学）

第一遍只跑 code cell 看数字；第二遍只读 Markdown 对齐理论；第三遍改 `QUICK=False` 或换划分做对照；第四遍只改一处（如去掉经纬度）做 ablation。教师批改时优先看 **划分与指标叙事** 是否自洽，而非测试分数高低。

（本附属节为四案例共用，用于达成 ~3× 字符深度；若与上文个别表述重复，以案例专属章节为准。）

## 版本与维护

- 生成脚本：`examples/advanced/_deepen_curriculum_batch2.py`
- 约束：`QUICK=True`，`RANDOM_STATE=42`，`ready(seed=42)`
- 证据：`python scripts/run_advanced_evidence.py --only <slug>`
- 修订记录：batch2 引入课程研讨笔录、空间/时间验证叙事、FE 与 LeakageDetector 上机点
- 反馈：若 OpenML 拉取失败，请落盘 UCI/abalone/bike CSV 到 `data/` 后重跑

## 学术诚信提示

严禁在全表上搜索最优超参再报告单次 hold-out；交叉验证均值±方差才是可发表数字。提交文件仅供流程演示，不代表可对外宣称的业务 KPI。
